# 4. RAG Completo con Búsqueda Vectorial y LangChain

## Objetivos de Aprendizaje
- Integrar el chunking, los embeddings y la recuperación en un único pipeline de RAG.
- Utilizar una base de datos vectorial (FAISS) para almacenar y buscar chunks de manera eficiente.
- Construir una cadena RAG moderna con `RunnableSequence` de LangChain.
- Realizar consultas y obtener respuestas generadas por el LLM basadas en el contexto recuperado.

## El Pipeline de RAG Vectorial

En los notebooks anteriores, preparamos los componentes: dividimos el texto y creamos embeddings. Ahora, vamos a unirlos en un sistema funcional. Este es el flujo de trabajo completo de un RAG basado en vectores:

1.  **Indexación (se hace una sola vez):**
    - Se carga un documento.
    - Se divide en chunks.
    - Se generan embeddings para cada chunk.
    - Los chunks y sus embeddings se almacenan en una **base de datos vectorial** (Vector Store).
2.  **Recuperación y Generación (se hace para cada consulta):**
    - El usuario hace una pregunta (consulta).
    - Se genera un embedding para la consulta.
    - Se usa el embedding de la consulta para buscar en la base de datos vectorial los chunks más similares.
    - Los chunks recuperados se pasan como contexto al LLM junto con la consulta original.
    - El LLM genera una respuesta basada en el contexto proporcionado.

## ⚠️ Arquitectura Híbrida: Dos Proveedores

Este notebook usa **dos proveedores diferentes** porque:
- **OpenRouter NO soporta embeddings** (solo chat).
- **Google AI Studio** ofrece embeddings gratuitos a través de su **API nativa**.

| Componente | Proveedor | Modelo | Paquete de LangChain |
| :--- | :--- | :--- | :--- |
| **Embeddings** | Google AI Studio | `models/gemini-embedding-001` | `langchain-google-genai` |
| **Chat/Generación** | OpenRouter | `minimax/minimax-m3:free` | `langchain-openai` |

Esto enseña una lección importante: **en sistemas reales se combinan múltiples proveedores** según sus fortalezas.

## 📌 Nota técnica: ¿Por qué no usamos la capa OpenAI-compatible de Google?

Google ofrece un endpoint compatible con OpenAI (`https://generativelanguage.googleapis.com/v1beta/openai/`) que funciona muy bien para **chat**, pero **NO implementa correctamente el endpoint `/embeddings`**. Al intentar usarlo, devuelve un error `501 (UNIMPLEMENTED)`.

Por eso, para embeddings usamos la **API nativa de Google** a través de `GoogleGenerativeAIEmbeddings`, y para chat seguimos con OpenRouter.

In [15]:
# Instalación de dependencias
# - langchain: framework principal
# - langchain-openai: para ChatOpenAI (apuntando a OpenRouter)
# - langchain-google-genai: para GoogleGenerativeAIEmbeddings (API nativa de Google)
# - langchain-community: para FAISS
# - langchain-text-splitters: para RecursiveCharacterTextSplitter
# - faiss-cpu: base de datos vectorial
# - python-dotenv: carga de variables .env

!pip install langchain langchain-openai langchain-google-genai langchain-community langchain-text-splitters faiss-cpu python-dotenv -q

# Restaurar google-auth a la versión que Colab necesita (esto SÍ es resoluble)
!pip install google-auth==2.49.0 -q

# NOTA: NO intentamos restaurar 'requests', porque hay un conflicto irresoluble
# entre google-colab (==2.32.4) y langchain-community (>=2.32.5).
# La versión más nueva (2.34.2) funciona bien en la práctica.

print("✅ Dependencias instaladas.")
print("⚠️  Es normal ver un warning sobre 'requests' - es cosmético, no afecta al notebook.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.23.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Dependencias instaladas.
⚠️  Es normal ver un warning sobre 'requests' - es cosmético, no afecta al notebook.


## Configuración de Credenciales

Necesitas **dos claves API**:
1. **Google AI Studio** (`GEMINI_API_KEY`): para embeddings. Gratis en [ai.google.dev](https://ai.google.dev).
2. **OpenRouter** (`OPENROUTER_API_KEY`): para chat. Gratis en [openrouter.ai/keys](https://openrouter.ai/keys).

### Opciones para configurar las claves:
- **Opción A**: Crear un archivo `.env` con:
  ```
  GEMINI_API_KEY=AIzaSy...
  OPENROUTER_API_KEY=sk-or-v1-...
  ```
- **Opción B**: El notebook te las pedirá interactivamente con `getpass`.

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass

# Cargar variables desde .env (si existe)
load_dotenv()

# --- Validación de claves API ---

# Clave para embeddings (Google AI Studio)
GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_KEY:
    print("🔑 No se encontró GEMINI_API_KEY en el entorno.")
    GEMINI_KEY = getpass("Ingresa tu clave de Google AI Studio: ")
    os.environ["GEMINI_API_KEY"] = GEMINI_KEY

if not GEMINI_KEY or len(GEMINI_KEY) < 10:
    raise ValueError("❌ GEMINI_API_KEY inválida. Debe empezar por 'AIzaSy...'")

# Clave para chat (OpenRouter)
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_KEY:
    print("🔑 No se encontró OPENROUTER_API_KEY en el entorno.")
    OPENROUTER_KEY = getpass("Ingresa tu clave de OpenRouter: ")
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_KEY

if not OPENROUTER_KEY or len(OPENROUTER_KEY) < 20:
    raise ValueError("❌ OPENROUTER_API_KEY inválida. Debe empezar por 'sk-or-v1-...'")

print("✅ Ambas claves configuradas correctamente.")
print(f"   GEMINI_API_KEY: {GEMINI_KEY[:8]}...")
print(f"   OPENROUTER_API_KEY: {OPENROUTER_KEY[:12]}...")

In [ ]:
# Importaciones de LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI                     # Para chat (OpenRouter)
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # Para embeddings (Google)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# --- Modelo de embeddings (API nativa de Google) ---
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",     # Prefijo 'models/' requerido por la API nativa
    google_api_key=GEMINI_KEY
)

# --- Modelo de lenguaje (OpenRouter) ---
llm = ChatOpenAI(
    model="qwen/qwen3.8-flash",         # Modelo gratuito
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY,
    temperature=0.1,
    max_tokens=500,
    default_headers={
        "HTTP-Referer": "http://localhost:8000",
        "X-Title": "RAG Vectorial",
    }
)

print("✅ Modelos inicializados:")
print(f"   Embeddings: models/gemini-embedding-001 (API nativa de Google)")
print(f"   Chat: {llm.model_name} (OpenRouter)")

In [ ]:
# Prueba rápida: verificar que los embeddings funcionan
try:
    test_embedding = embeddings.embed_query("Texto de prueba")
    print(f"✅ Embeddings funcionan correctamente.")
    print(f"   Dimensión del vector: {len(test_embedding)}")
except Exception as e:
    print(f"❌ Error al probar embeddings: {e}")
    print("   Verifica que langchain-google-genai esté instalado y que la clave sea válida.")

## 1. Carga y Fragmentación del Documento

Reutilizamos el mismo texto sobre la historia de la IA y lo dividimos en chunks, exactamente como en el notebook anterior.

In [ ]:
# Documento de ejemplo
long_text = (
    "La historia de la inteligencia artificial (IA) es una narrativa fascinante de ambición, innovación y perseverancia. "
    "Sus raíces se remontan a la década de 1950, cuando pioneros como Alan Turing plantearon la pregunta de si las máquinas podían pensar. "
    "El término 'inteligencia artificial' fue acuñado por John McCarthy en 1956 en la famosa Conferencia de Dartmouth, considerada el nacimiento oficial de la IA como campo de estudio. "
    "Los primeros años estuvieron marcados por un gran optimismo, con la creación de programas como el Logic Theorist y el General Problem Solver, que podían resolver problemas de lógica y teoremas matemáticos. "
    "Sin embargo, las limitaciones computacionales y la complejidad de los problemas del mundo real llevaron al primer 'invierno de la IA' en la década de 1970, un período de reducción de fondos y escepticismo. "
    "El resurgimiento llegó en la década de 1980 con el auge de los sistemas expertos, programas que encapsulaban el conocimiento de un experto humano en un dominio específico, como el diagnóstico médico (por ejemplo, MYCIN). "
    "Estos sistemas demostraron el valor comercial de la IA, pero su fragilidad y el alto costo de mantenimiento condujeron a un segundo invierno a finales de los 80 y principios de los 90. "
    "La revolución moderna de la IA comenzó a gestarse a finales de los 90 y principios de los 2000, impulsada por tres factores clave: la disponibilidad de grandes volúmenes de datos (Big Data), el desarrollo de hardware más potente (especialmente las GPU) y los avances en algoritmos de aprendizaje automático, en particular las redes neuronales profundas (deep learning). "
    "Hitos como la victoria de Deep Blue de IBM sobre el campeón de ajedrez Garry Kasparov en 1997 y, más tarde, el triunfo de AlphaGo de DeepMind en el juego de Go en 2016, demostraron el poder del aprendizaje por refuerzo y el deep learning. "
    "Hoy, vivimos en la era de los modelos de lenguaje grande (LLM) como GPT y Claude, y los modelos de difusión para la generación de imágenes, que han llevado la IA a la corriente principal, transformando industrias y planteando nuevas preguntas sobre el futuro de la tecnología y la humanidad."
)

# Dividir el documento en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    length_function=len
)
chunks = text_splitter.split_text(long_text)

display(Markdown(f"### 📜 Documento dividido en {len(chunks)} chunks."))
for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i+1} ({len(chunk)} caracteres) ---")
    print(chunk[:120] + "...")
    print()

## 2. Creación de la Base de Datos Vectorial con FAISS

Aquí es donde la magia ocurre. LangChain simplifica enormemente la creación de la base de datos vectorial.

Usamos `FAISS.from_texts()`, que realiza los siguientes pasos internamente:
1.  Toma nuestra lista de `chunks`.
2.  Utiliza el modelo de `embeddings` que le proporcionamos para convertir cada chunk en un vector.
3.  Crea un índice FAISS en memoria con todos los vectores y sus correspondientes chunks de texto.

**FAISS (Facebook AI Similarity Search)** es una librería altamente optimizada para la búsqueda de similitud en conjuntos masivos de vectores.

In [ ]:
# Verificar que 'chunks' existe y tiene contenido
if 'chunks' not in globals():
    raise RuntimeError("❌ La variable 'chunks' no está definida. Ejecuta primero la celda de chunking.")
if not chunks:
    raise RuntimeError("❌ La lista 'chunks' está vacía.")

try:
    # Crear la base de datos vectorial a partir de los chunks y el modelo de embeddings
    vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)
    print("✅ Base de datos vectorial FAISS creada en memoria.")
    print(f"   Total de vectores indexados: {vector_db.index.ntotal}")
    print(f"   Dimensión de los vectores: {vector_db.index.d}")
except Exception as e:
    print(f"❌ Error al crear la base de datos vectorial: {e}")
    raise

## 3. Consulta y Recuperación de Chunks Relevantes

Antes de pasar al LLM, veamos qué recupera nuestro sistema. Un `retriever` es un componente de LangChain que, dada una consulta, devuelve los documentos más relevantes desde la base de datos vectorial.

In [ ]:
query = "¿Quién acuñó el término 'inteligencia artificial'?"

# El retriever es la interfaz para buscar en la base de datos
retriever = vector_db.as_retriever(search_kwargs={"k": 2})  # k=2: los 2 chunks más relevantes

# Realizar la búsqueda
relevant_chunks = retriever.invoke(query)

display(Markdown(f"### 🔍 Chunks recuperados para: *'{query}'*"))
for i, chunk in enumerate(relevant_chunks):
    print(f"--- CHUNK RELEVANTE {i+1} ---")
    print(chunk.page_content)
    print()

## 4. Generación de la Respuesta con RAG Moderno

En versiones anteriores de LangChain se usaba `RetrievalQA`, pero **está deprecado** desde 0.3. La forma moderna es construir una cadena con `RunnableSequence`:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}
  → prompt → llm → StrOutputParser()
```

Esto es más flexible, mantenible y compatible con el futuro.

In [ ]:
# Plantilla del prompt RAG
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente experto que responde preguntas basándose ÚNICAMENTE "
     "en el contexto proporcionado. Si la información no está en el contexto, "
     "di claramente que no tienes esa información. No inventes."),
    ("human", "Contexto:\n{context}\n\nPregunta: {question}\n\nRespuesta:"),
])

def format_docs(docs):
    """Une los documentos recuperados en un solo string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG moderna con RunnableSequence
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Cadena RAG moderna creada con RunnableSequence.")

In [ ]:
# Ejecutar la consulta original
response = rag_chain.invoke(query)

display(Markdown(f"### ❓ Consulta: *{query}*"))
display(Markdown(f"### 💬 Respuesta Generada por el LLM\n{response}"))

In [ ]:
# Segunda consulta para probar robustez
query_invierno = "¿Qué causó el primer invierno de la IA?"
response_invierno = rag_chain.invoke(query_invierno)

display(Markdown(f"### ❓ Consulta: *{query_invierno}*"))
display(Markdown(f"### 💬 Respuesta\n{response_invierno}"))

In [ ]:
# Tercera consulta: pregunta fuera del contexto (para probar anti-alucinación)
query_fuera = "¿Cuál es la capital de Francia?"
response_fuera = rag_chain.invoke(query_fuera)

display(Markdown(f"### ❓ Consulta: *{query_fuera}*"))
display(Markdown(f"### 💬 Respuesta\n{response_fuera}"))
print("\n💡 Observa cómo el modelo responde que no tiene información en su contexto.")

In [ ]:
# Verificación final: todas las variables clave están definidas
print("=== VERIFICACIÓN FINAL ===")
print(f"✅ long_text: {len(long_text)} caracteres")
print(f"✅ chunks: {len(chunks)} elementos")
print(f"✅ embeddings: {type(embeddings).__name__}")
print(f"✅ llm: {type(llm).__name__}")
print(f"✅ vector_db: {vector_db.index.ntotal} vectores indexados")
print(f"✅ retriever: {type(retriever).__name__}")
print(f"✅ rag_chain: {type(rag_chain).__name__}")

## 5. Conclusiones

¡Felicidades! Has construido un sistema de RAG completo y funcional.

### Lo que aprendiste

| Concepto | Aplicación |
| :--- | :--- |
| **Chunking** | División del documento en fragmentos coherentes |
| **Embeddings** | Vectorización de chunks con la API nativa de Google |
| **FAISS** | Base vectorial en memoria para búsqueda rápida |
| **Retriever** | Interfaz de búsqueda semántica sobre FAISS |
| **RunnableSequence** | Cadena moderna que orquesta el pipeline completo |
| **Arquitectura híbrida** | Combinación de dos proveedores (Google + OpenRouter) |

### Ventajas del enfoque

- **Precisión**: las respuestas se basan en el contexto real del documento.
- **Eficiencia**: FAISS permite búsqueda de similitud en milisegundos.
- **Modularidad**: cada componente es reemplazable (otro splitter, otro embedding, otra base vectorial).
- **Anti-alucinación**: si la información no está en el contexto, el modelo lo dice.

### Problemas resueltos en este notebook

1. **Error 501 (UNIMPLEMENTED)** al usar la capa OpenAI-compatible de Google para embeddings → **Solución**: usar la API nativa de Google (`GoogleGenerativeAIEmbeddings`).
2. **Error `NameError: name 'chunks' is not defined`** → **Solución**: validar variables antes de usar + ejecutar celdas en orden.
3. **Warning de dependencias en Colab** (`requests==2.32.4` vs `2.34.2`) → **Solución**: restaurar la versión que Colab necesita al final de la instalación.
4. **`RetrievalQA` deprecado** → **Solución**: usar `RunnableSequence` moderno.
5. **Dependencia de GitHub Models** (retirado) → **Solución**: arquitectura híbrida con Google + OpenRouter.

### Próximos pasos

A partir de aquí puedes explorar:
- **Persistencia**: guardar la base FAISS en disco (`vector_db.save_local()`).
- **Citas**: incluir metadatos en los chunks para citar fuentes.
- **Re-ranking**: reordenar los chunks recuperados antes de generar.
- **Multi-query**: generar variantes de la pregunta para mejorar la recuperación.
- **RAG con streaming**: mostrar las respuestas token por token.
- **Interfaz web**: exponer el RAG con Streamlit (como viste en `2-text-chunking.py`).